## Context & Methods

### Key assumptions

- `q34` takes priority over the existing `lsoa_name` value.
- A matched postcode outside Exeter is classified as `21.0 = Cranbrook`.
- The existing code is used only to construct a postcode consensus baseline. Reviewed overrides resolve conflicting postcodes and clear geographic outliers.
- Partial/unmatched postcodes and documented boundary cases remain marked uncertain instead of being silently treated as confirmed.

In [10]:
from pathlib import Path
import csv
import re

import pandas as pd

DATA_DIR = Path('.')
INPUT_CSV = DATA_DIR / 'data_raw__no_index.csv'
ONSPD_CSV = DATA_DIR / 'ONSPD_MAY_2026_UK_EX.csv'
LSOA_LOOKUP_CSV = DATA_DIR / 'LSOA_(2021)_to_Electoral_Ward_(2024)_to_LAD_(2024)_Best_Fit_Lookup_in_EW.csv'
#OUTPUT_CSV = DATA_DIR / 'data_raw__no_index_area_classified_main.csv'
OUTPUT_CSV = DATA_DIR / 'data_raw__no_index.csv'

AREA_LABELS = {
    '1.0': 'Exeter: Alphington Road - Ebrington Road area',
    '2.0': 'Exeter: Beacon Heath area',
    '3.0': 'Exeter: Burnthouse Lane area (Poets)',
    '4.0': 'Exeter: Burnthouse Lane area (Rifford Road)',
    '5.0': 'Exeter: Burnthouse Lane area (Trees)',
    '6.0': 'Exeter: Cathedral and City Centre East',
    '7.0': 'Exeter: City Centre West',
    '8.0': 'Exeter: Countess Wear, Lower Wear',
    '9.0': 'Exeter: Cowick - Barley Farm Road area',
    '10.0': 'Exeter: Cowick - Newman Road area',
    '11.0': 'Exeter: Exwick - Cemetery area',
    '12.0': 'Exeter: Exwick - Farm Hill area',
    '13.0': 'Exeter: Exwick - Redhills area',
    '14.0': 'Exeter: Hamlins Lane and Honeylands area',
    '15.0': 'Exeter: Lancelot Road area',
    '16.0': 'Exeter: Sidwell Street and Clifton Road area',
    '17.0': 'Exeter: Stoke Hill',
    '18.0': 'Exeter: Summerway area',
    '19.0': 'Exeter: Whipton - Leypark Road and Bramley Avenue area',
    '20.0': 'Exeter: Whipton, Hillyfield Road area',
    '21.0': 'Cranbrook',
}

BROAD_AREA_BY_CODE = {
    '1.0': 'St Thomas',
    '2.0': 'Beacon Heath',
    '3.0': 'Wonford',
    '4.0': 'Wonford',
    '5.0': 'Wonford',
    '6.0': 'City Centre',
    '7.0': "St David's",
    '8.0': 'Countess Wear',
    '9.0': 'St Thomas',
    '10.0': 'St Thomas',
    '11.0': 'Exwick',
    '12.0': 'Exwick',
    '13.0': 'Redhills',
    '14.0': 'Mincinglake',
    '15.0': 'Beacon Heath',
    '16.0': 'Newtown',
    '17.0': 'Mincinglake',
    '18.0': 'Whipton',
    '19.0': 'Whipton',
    '20.0': 'Whipton',
    '21.0': 'Cranbrook',
}

assert set(BROAD_AREA_BY_CODE) == set(AREA_LABELS), 'Broad-area mapping must cover all 21 detailed codes'

In [11]:
# Reviewed after exact-postcode conflicts, ONSPD coordinates/LSOAs, nearby postcode
# consensus, and street/target-area names were compared. Keys omit spaces.
POSTCODE_OVERRIDES = {
    'EX28JQ': '1.0',
    'EX28BQ': '1.0',
    'EX48LL': '2.0',
    'EX48LX': '2.0',
    'EX48LS': '2.0',
    'EX49AA': '2.0',
    'EX49AB': '2.0',
    'EX47JW': '2.0',
    'EX24SW': '3.0',
    'EX26AH': '3.0',
    'EX26BG': '3.0',
    'EX26BH': '3.0',
    'EX26DB': '3.0',
    'EX25JH': '4.0',
    'EX25JT': '4.0',
    'EX26BQ': '5.0',
    'EX26DH': '5.0',
    'EX42BZ': '11.0',
    'EX41SW': '13.0',
    'EX49AG': '15.0',
    'EX28JE': '1.0',
}

POSTCODE_OVERRIDE_REASONS = {
    'EX25JH': 'Rifford Road-area postcode; fixes spreadsheet row 925',
    'EX25JT': 'Rifford Road/Wonford Community Association postcode; fixes spreadsheet row 146',
    'EX28JE': 'Alphington Road-area postcode; fixes spreadsheet row 1525',
    'EX49AG': 'Merlin Crescent is in the Lancelot Road/Arthurian-streets area',
    'EX26DB': 'Milton Road belongs to the Poets area',
    'EX26DH': 'Chestnut Avenue belongs to the Trees area',
    'EX41SW': 'Ashleigh Mount Road is in Redhills',
    'EX42BZ': 'Exwick Road postcode aligns with the Cemetery area',
    'EX49AB': 'Tor Close is in Beacon Heath',
    'EX47JW': 'Nearest reviewed postcode consensus places this part of St Katherines Road in area 2',
}

REVIEW_UNCERTAIN_POSTCODES = {
    'EX26AT': 'Burnthouse Lane boundary postcode: exact Poets/Trees subarea cannot be confirmed without the project boundary polygon',
    'EX47JW': 'Postcode is close to the Beacon Heath/Stoke Hill boundary; assigned to area 2 from nearest postcode consensus',
}

In [12]:
def normalize_postcode_key(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', '', str(value).strip().upper())


def normalize_area_code(value):
    if pd.isna(value):
        return ''
    text = str(value).strip()
    match = re.fullmatch(r'(\d+)(?:\.0+)?', text)
    return f'{int(match.group(1))}.0' if match else text


def append_reason(existing, reason):
    return reason if not existing else f'{existing}; {reason}'

## Data

Load the questionnaire, ONSPD postcode geography, and the 2021 LSOA-to-ward/LAD lookup. Joins are left joins and row-count preservation is tested later.

In [13]:
source = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
source['original_area_code'] = source['lsoa_name'].map(normalize_area_code)
source['postcode_key'] = source['q34'].map(normalize_postcode_key)

onspd = pd.read_csv(
    ONSPD_CSV,
    dtype=str,
    keep_default_na=False,
    usecols=['pcds', 'lsoa21cd'],
)
onspd['postcode_key'] = onspd['pcds'].map(normalize_postcode_key)
onspd = (
    onspd.loc[onspd['postcode_key'].ne(''), ['postcode_key', 'pcds', 'lsoa21cd']]
    .drop_duplicates('postcode_key')
    .rename(columns={
        'pcds': 'postcode_clean',
        'lsoa21cd': 'postcode_lsoa21cd',
    })
)

lsoa_lookup = pd.read_csv(
    LSOA_LOOKUP_CSV,
    dtype=str,
    keep_default_na=False,
    usecols=['LSOA21CD', 'LSOA21NM', 'WD24NM', 'LAD24NM'],
).rename(columns={
    'LSOA21CD': 'postcode_lsoa21cd',
    'LSOA21NM': 'postcode_lsoa21nm',
    'WD24NM': 'postcode_ward24nm',
    'LAD24NM': 'postcode_lad24nm',
})

classified = source.merge(onspd, on='postcode_key', how='left', validate='many_to_one')
classified = classified.merge(lsoa_lookup, on='postcode_lsoa21cd', how='left', validate='many_to_one')
lookup_columns = [
    'postcode_clean', 'postcode_lsoa21cd', 'postcode_lsoa21nm',
    'postcode_ward24nm', 'postcode_lad24nm',
]
classified[lookup_columns] = classified[lookup_columns].fillna('')
classified['postcode_lookup_status'] = 'matched'
classified.loc[classified['postcode_key'].eq(''), 'postcode_lookup_status'] = 'blank'
classified.loc[
    classified['postcode_key'].ne('') & classified['postcode_clean'].eq(''),
    'postcode_lookup_status',
] = 'not_found'

print(f'Rows: {len(classified):,}; columns: {classified.shape[1]:,}')
classified['postcode_lookup_status'].value_counts(dropna=False)

Rows: 1,664; columns: 169


postcode_lookup_status
matched      1662
not_found       2
Name: count, dtype: int64

In [14]:
# Build a deterministic postcode baseline from the modal existing code.
# Reviewed overrides replace every observed tie/conflict and known geographic outlier.
postcode_votes = (
    source.loc[source['original_area_code'].isin(AREA_LABELS) & source['postcode_key'].ne('')]
    .groupby(['postcode_key', 'original_area_code'])
    .size()
    .reset_index(name='records')
)
postcode_votes['area_sort'] = pd.to_numeric(postcode_votes['original_area_code'], errors='coerce')
postcode_votes = postcode_votes.sort_values(
    ['postcode_key', 'records', 'area_sort'],
    ascending=[True, False, True],
)
postcode_area_map = (
    postcode_votes.drop_duplicates('postcode_key')
    .set_index('postcode_key')['original_area_code']
    .to_dict()
)
postcode_area_map.update(POSTCODE_OVERRIDES)

conflicting_postcodes = (
    postcode_votes.groupby('postcode_key')['original_area_code'].nunique()
    .loc[lambda count: count > 1]
    .index
)
unresolved_conflicting_postcodes = sorted(set(conflicting_postcodes) - set(POSTCODE_OVERRIDES))
assert not unresolved_conflicting_postcodes, (
    'Add reviewed overrides for conflicting postcodes: ' + ', '.join(unresolved_conflicting_postcodes)
)
print(f'Resolved conflicting postcodes: {len(conflicting_postcodes):,}')

Resolved conflicting postcodes: 14


In [15]:
classified['area_interest_code'] = classified['postcode_key'].map(postcode_area_map).fillna('')
classified['area_assignment_source'] = 'authoritative_postcode_consensus'

matched_postcode = classified['postcode_lookup_status'].eq('matched')
matched_non_exeter = matched_postcode & classified['postcode_lad24nm'].ne('Exeter')
classified.loc[matched_non_exeter, 'area_interest_code'] = '21.0'
classified.loc[matched_non_exeter, 'area_assignment_source'] = 'authoritative_non_exeter_postcode'

reviewed_override = classified['postcode_key'].isin(POSTCODE_OVERRIDES)
classified.loc[reviewed_override, 'area_interest_code'] = classified.loc[reviewed_override, 'postcode_key'].map(POSTCODE_OVERRIDES)
classified.loc[reviewed_override, 'area_assignment_source'] = 'reviewed_postcode_override'

# A partial/unmatched q34 cannot support an exact postcode classification.
# Retain a valid original code only as a visibly uncertain fallback.
unmatched = ~matched_postcode
valid_original = classified['original_area_code'].isin(AREA_LABELS)
classified.loc[unmatched & valid_original, 'area_interest_code'] = classified.loc[
    unmatched & valid_original, 'original_area_code'
]
classified.loc[unmatched & valid_original, 'area_assignment_source'] = 'uncertain_original_code_fallback'
classified.loc[classified['area_interest_code'].eq(''), 'area_assignment_source'] = 'unassigned'

classified['area_interest_name'] = classified['area_interest_code'].map(AREA_LABELS).fillna('')
classified['broad_area_name'] = classified['area_interest_code'].map(BROAD_AREA_BY_CODE).fillna('')
classified['area_classification_changed'] = (
    classified['area_interest_code'].ne(classified['original_area_code'])
)
classified['area_correction_reason'] = classified['postcode_key'].map(POSTCODE_OVERRIDE_REASONS).fillna('')

In [16]:
classified['area_uncertainty_reason'] = ''

for index, row in classified.iterrows():
    reason = ''
    if row['area_interest_code'] not in AREA_LABELS:
        reason = append_reason(reason, 'area code not in 1.0-21.0')
    if row['postcode_lookup_status'] == 'blank':
        reason = append_reason(reason, 'q34 postcode blank')
    elif row['postcode_lookup_status'] == 'not_found':
        reason = append_reason(reason, 'q34 is partial or not found in ONSPD')
    if row['postcode_lookup_status'] == 'matched' and not row['postcode_lad24nm']:
        reason = append_reason(reason, 'postcode matched but LAD lookup is missing')
    if row['postcode_key'] in REVIEW_UNCERTAIN_POSTCODES:
        reason = append_reason(reason, REVIEW_UNCERTAIN_POSTCODES[row['postcode_key']])
    if row['postcode_lookup_status'] == 'matched' and row['postcode_lad24nm'] == 'Exeter' and row['area_interest_code'] == '21.0':
        reason = append_reason(reason, 'Exeter postcode is still classified as Cranbrook')
    if row['postcode_lookup_status'] == 'matched' and row['postcode_lad24nm'] != 'Exeter' and row['area_interest_code'] != '21.0':
        reason = append_reason(reason, 'non-Exeter postcode is not classified as Cranbrook')
    classified.at[index, 'area_uncertainty_reason'] = reason

classified['area_classification_uncertain'] = classified['area_uncertainty_reason'].ne('')
classified['area_validation_note'] = classified['area_uncertainty_reason']

## Results

Run the checks below before saving. They prevent row loss, duplicate IDs, inconsistent final codes for the same complete postcode, or Exeter/Cranbrook contradictions.

In [17]:
assert len(classified) == len(source), 'Join changed the questionnaire row count'
assert classified['id.case'].is_unique, 'id.case is not unique after classification'
assert classified['area_interest_code'].isin(AREA_LABELS).all(), 'Some records remain unassigned'
assert classified['broad_area_name'].ne('').all(), 'Some records have no broad-area assignment'
assert not {'postcode_lat', 'postcode_long'} & set(classified.columns), 'Coordinate columns must not be created'
assert (
    classified.groupby('area_interest_code')['broad_area_name'].nunique().le(1).all()
), 'A detailed area code maps to more than one broad area'

complete_matched = classified['postcode_lookup_status'].eq('matched')
final_postcode_conflicts = (
    classified.loc[complete_matched]
    .groupby('postcode_key')['area_interest_code']
    .nunique()
    .loc[lambda count: count > 1]
)
assert final_postcode_conflicts.empty, 'Identical postcodes have different final area codes'

exeter_to_cranbrook = complete_matched & classified['postcode_lad24nm'].eq('Exeter') & classified['area_interest_code'].eq('21.0')
non_exeter_to_exeter_area = complete_matched & classified['postcode_lad24nm'].ne('Exeter') & classified['area_interest_code'].ne('21.0')
assert not exeter_to_cranbrook.any(), 'At least one Exeter postcode is classified as Cranbrook'
assert not non_exeter_to_exeter_area.any(), 'At least one non-Exeter postcode is classified in areas 1-20'

cited_rows = classified.iloc[[144, 923, 1523]][
    ['id.case', 'q34', 'original_area_code', 'area_interest_code', 'area_interest_name']
]
cited_rows.index = [146, 925, 1525]  # spreadsheet row numbers including the header
cited_rows

,id.case,q34,original_area_code,area_interest_code,area_interest_name
146,159b06ed-fcad-46d2-8d38-52e307e0f8ff,EX2 5JT,21.0,4.0,Exeter: Burnthouse Lane area (Rifford Road)
925,8a990069-daab-4c3e-899b-41e0b3375fe4,EX2 5JH,21.0,4.0,Exeter: Burnthouse Lane area (Rifford Road)
1525,e911e80e-7657-4fe2-b4f1-17288a38922f,EX2 8JE,21.0,1.0,Exeter: Alphington Road - Ebrington Road area


In [18]:
output = classified.drop(columns=['postcode_key'])
output.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding='utf-8',
    quoting=csv.QUOTE_MINIMAL,
    lineterminator='\n',
)

summary = pd.Series({
    'source_rows': len(source),
    'output_rows': len(output),
    'changed_records': int(output['area_classification_changed'].sum()),
    'uncertain_records': int(output['area_classification_uncertain'].sum()),
    'postcode_lookup_failures': int(output['postcode_lookup_status'].ne('matched').sum()),
    'broad_areas': int(output['broad_area_name'].nunique()),
})
print(f'Wrote {OUTPUT_CSV}')
display(summary.to_frame('value'))

changed_records = output.loc[
    output['area_classification_changed'],
    ['id.case', 'q34', 'original_area_code', 'area_interest_code', 'area_interest_name', 'broad_area_name', 'area_correction_reason'],
]
uncertain_records = output.loc[
    output['area_classification_uncertain'],
    ['id.case', 'q34', 'area_interest_code', 'area_interest_name', 'broad_area_name', 'area_uncertainty_reason'],
]
display(changed_records)
display(uncertain_records)

Wrote data_raw__no_index.csv


,value
source_rows,1664
output_rows,1664
changed_records,34
uncertain_records,4
postcode_lookup_failures,2
broad_areas,12


,id.case,q34,original_area_code,area_interest_code,area_interest_name,broad_area_name,area_correction_reason
44,06a6fe6c-457f-40f2-a9bc-8062eb5e32f7,EX2 4SW,2.0,3.0,Exeter: Burnthouse Lane area (Poets),Wonford,
53,07cfd076-2460-4b10-89e9-be3b2cf80a13,EX2 6BG,9.0,3.0,Exeter: Burnthouse Lane area (Poets),Wonford,
73,0bba25c4-11b5-4b35-a080-315cd899a195,EX4 9AA,16.0,2.0,Exeter: Beacon Heath area,Beacon Heath,
96,0dd049c4-8c57-4b33-be45-7cffe0a23aa6,EX4 8LS,11.0,2.0,Exeter: Beacon Heath area,Beacon Heath,
122,1262c368-38dd-42ee-905c-8948c9370a05,EX4 9AG,19.0,15.0,Exeter: Lancelot Road area,Beacon Heath,Merlin Crescent is in the Lancelot Road/Arthur...
126,12e2087f-61b3-491b-ad77-473ea3c5a6ff,EX2 6BQ,3.0,5.0,Exeter: Burnthouse Lane area (Trees),Wonford,
144,159b06ed-fcad-46d2-8d38-52e307e0f8ff,EX2 5JT,21.0,4.0,Exeter: Burnthouse Lane area (Rifford Road),Wonford,Rifford Road/Wonford Community Association pos...
285,29635975-0be3-4fa6-bdce-d8b1b729bc33,EX4 2BZ,13.0,11.0,Exeter: Exwick - Cemetery area,Exwick,Exwick Road postcode aligns with the Cemetery ...
341,30bdc614-821f-4776-b77a-99d57523854e,EX2 6DH,4.0,5.0,Exeter: Burnthouse Lane area (Trees),Wonford,Chestnut Avenue belongs to the Trees area
371,361f97f9-b4a6-4cd2-acfa-96e3ad4cbef8,EX4 8LL,8.0,2.0,Exeter: Beacon Heath area,Beacon Heath,


,id.case,q34,area_interest_code,area_interest_name,broad_area_name,area_uncertainty_reason
209,1e130e68-e28e-441f-a64a-dab613b9b7ee,EX1 1,6.0,Exeter: Cathedral and City Centre East,City Centre,q34 is partial or not found in ONSPD
283,2956d457-afd4-4596-a106-0915f8b75f52,EX5 7,21.0,Cranbrook,Cranbrook,q34 is partial or not found in ONSPD
1149,b131cbf5-1708-4945-bf11-5e181e1184c5,EX2 6AT,3.0,Exeter: Burnthouse Lane area (Poets),Wonford,Burnthouse Lane boundary postcode: exact Poets...
1615,f79b914e-36bc-4f64-92d1-8f574dab2909,EX4 7JW,2.0,Exeter: Beacon Heath area,Beacon Heath,Postcode is close to the Beacon Heath/Stoke Hi...


## Takeaways

The final `area_interest_code` and `area_interest_name` retain the detailed 21-area classification. `broad_area_name` supplies the broader neighbourhood grouping. `original_area_code` preserves the incoming `lsoa_name` value, `area_classification_changed` identifies corrections, and `area_classification_uncertain` identifies records that still require caution.